# Cálculo del Índice de Calidad de Detección (ICD) - Datos 2026

**Objetivo:** Calcular el ICD para las simulaciones 2026 y comparar con resultados previos.

**Fórmula del ICD:**

$$
\text{ICD} = \underbrace{\text{DetOK}}_{\text{Componente D}} \times \underbrace{\frac{\ln(1 + \alpha \times \%)}{\ln(1 + \alpha \times \%_{\max})}}_{\text{Componente } C_{\text{norm}}} \times \underbrace{e^{-\beta \times N_{\text{FP}}}}_{\text{Penalización } P_{\text{FP}}}
$$

**Donde:**
- **DetOK:** Éxito de detección (0, 0.5, 1)
  - `0`: Fallo completo
  - `0.5`: Detección indirecta (nodos cercanos)
  - `1`: Detección directa
- **C_norm:** Confianza normalizada (penaliza detección en severidades bajas)
  - `α = 0.1` (factor de ajuste)
  - `%_max = 45%` para abolladuras, `%_max = 90%` para corrosión
- **P_FP:** Penalización exponencial por falsos positivos
  - `β = 0.15` (factor de penalización)
  - `N_FP`: Número de falsos positivos

**Referencias:**
- Pipeline original: `outputs/resultados_antiguos/resultados_abolladuras/jupyter_notebooks/analisis_datos_abolladura.ipynb`
- Documentación: `outputs/Resultados/registro_pipeline_danos.md`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configuración de gráficas
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("✓ Librerías cargadas")

## 1. Configuración y Carga de Datos

In [ ]:
# Rutas absolutas
base_path = Path.home() / 'github' / 'Proyecto-doctoral'
resultados_nuevos = base_path / 'outputs' / 'resultados_nuevos'

# Selección de tipo de daño
TIPO_DANO = 'abolladura'  # Cambiar a 'corrosion' para procesar corrosión

# Rutas según tipo de daño
if TIPO_DANO == 'abolladura':
    df_path = resultados_nuevos / 'abolladuras_2026' / 'todos_los_resultados_remapeado_final.xlsx'
    output_dir = resultados_nuevos / 'abolladuras_2026'
    porcentaje_max = 45  # [%] Severidad máxima en abolladuras
elif TIPO_DANO == 'corrosion':
    df_path = resultados_nuevos / 'corrosion_2026' / 'todos_los_resultados_remapeado_final.xlsx'
    output_dir = resultados_nuevos / 'corrosion_2026'
    porcentaje_max = 90  # [%] Severidad máxima en corrosión
else:
    raise ValueError(f"Tipo de daño '{TIPO_DANO}' no reconocido")

# Parámetros del ICD (validados en análisis previos)
ALPHA = 0.1   # Factor de ajuste para confianza
BETA = 0.15   # Factor de penalización por falsos positivos

print(f"\n{'='*70}")
print(f"CONFIGURACIÓN - {TIPO_DANO.upper()}")
print(f"{'='*70}")
print(f"Dataset: {df_path.name}")
print(f"Existe: {df_path.exists()}")
print(f"\nParámetros del ICD:")
print(f"  α (confianza): {ALPHA}")
print(f"  β (penalización): {BETA}")
print(f"  %_max: {porcentaje_max}%")

In [ ]:
# Cargar dataset remapeado
print("\n📂 Cargando dataset remapeado...")
df = pd.read_excel(df_path, engine='openpyxl')

print(f"✓ Dimensiones: {df.shape}")
print(f"✓ Columnas: {list(df.columns)}")
print(f"\n📊 Primeras 5 filas:")
display(df.head())

# Verificar presencia de columnas requeridas
required_cols = ['ID', 'Elemento', 'Porcentaje', 'DeteccionOK', 'N_FalsosPositivos']
missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"Columnas faltantes: {missing_cols}")

print(f"\n✓ Todas las columnas requeridas presentes")

## 2. Cálculo del ICD

In [ ]:
def calcular_ICD(df, alpha, beta, porcentaje_max):
    """
    Calcula el Índice de Calidad de Detección (ICD).
    
    ICD = DetOK × [ln(1 + α×%) / ln(1 + α×%_max)] × exp(-β × N_FP)
    
    Parámetros:
    -----------
    df : DataFrame
        Dataset con columnas: Porcentaje, DeteccionOK, N_FalsosPositivos
    alpha : float
        Factor de ajuste para confianza (típicamente 0.1)
    beta : float
        Factor de penalización por falsos positivos (típicamente 0.15)
    porcentaje_max : float
        Severidad máxima del tipo de daño (%)
    
    Retorna:
    --------
    DataFrame : Dataset original con columnas adicionales:
        - C_norm: Componente de confianza normalizada
        - P_FP: Penalización por falsos positivos
        - ICD: Índice de Calidad de Detección
    """
    df_icd = df.copy()
    
    # Componente 1: Éxito de detección (D)
    # Ya está en DeteccionOK (0, 0.5, 1)
    D = df_icd['DeteccionOK']
    
    # Componente 2: Confianza normalizada (C_norm)
    # Penaliza detecciones en severidades bajas
    C_norm = np.log(1 + alpha * df_icd['Porcentaje']) / np.log(1 + alpha * porcentaje_max)
    
    # Componente 3: Penalización por falsos positivos (P_FP)
    # Decae exponencialmente con el número de FPs
    P_FP = np.exp(-beta * df_icd['N_FalsosPositivos'])
    
    # Calcular ICD
    ICD = D * C_norm * P_FP
    
    # Agregar columnas al DataFrame
    df_icd['C_norm'] = C_norm
    df_icd['P_FP'] = P_FP
    df_icd['ICD'] = ICD
    
    return df_icd


# Aplicar cálculo
print(f"\n💻 Calculando ICD...")
df_icd = calcular_ICD(df, alpha=ALPHA, beta=BETA, porcentaje_max=porcentaje_max)

print(f"✓ ICD calculado para {len(df_icd)} corridas")
print(f"\n📊 Estadísticas del ICD:")
print(df_icd['ICD'].describe())

# Mostrar ejemplos
print(f"\n📋 Ejemplos de ICD calculado:")
cols_mostrar = ['ID', 'Elemento', 'Porcentaje', 'DeteccionOK', 'N_FalsosPositivos', 'C_norm', 'P_FP', 'ICD']
display(df_icd[cols_mostrar].head(10))

## 3. Análisis por Severidad

In [ ]:
# Agrupar por severidad
icd_por_severidad = df_icd.groupby('Porcentaje')['ICD'].agg(['mean', 'std', 'min', 'max', 'count']).reset_index()
icd_por_severidad.columns = ['Porcentaje', 'ICD_mean', 'ICD_std', 'ICD_min', 'ICD_max', 'n_corridas']

print(f"\n📊 ICD promedio por severidad:")
display(icd_por_severidad)

# Gráfica: ICD vs Severidad
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(icd_por_severidad['Porcentaje'], icd_por_severidad['ICD_mean'], 
        marker='o', linewidth=2, markersize=8, label='ICD promedio')
ax.fill_between(icd_por_severidad['Porcentaje'], 
                icd_por_severidad['ICD_mean'] - icd_por_severidad['ICD_std'],
                icd_por_severidad['ICD_mean'] + icd_por_severidad['ICD_std'],
                alpha=0.3, label='± 1σ')

ax.set_xlabel('Severidad del daño (%)', fontsize=12)
ax.set_ylabel('ICD', fontsize=12)
ax.set_title(f'Índice de Calidad de Detección vs Severidad - {TIPO_DANO.capitalize()}', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / f'ICD_vs_severidad_{TIPO_DANO}.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✓ Gráfica guardada: ICD_vs_severidad_{TIPO_DANO}.png")

## 4. Análisis por Tipo de Elemento (si aplica)

In [ ]:
# Verificar si existe columna de tipo de elemento
if 'Tipo_elemento_a_buscar' in df_icd.columns:
    icd_por_tipo = df_icd.groupby(['Tipo_elemento_a_buscar', 'Porcentaje'])['ICD'].mean().reset_index()
    
    print(f"\n📊 ICD promedio por tipo de elemento:")
    
    fig, ax = plt.subplots(figsize=(14, 6))
    
    for tipo in icd_por_tipo['Tipo_elemento_a_buscar'].unique():
        data = icd_por_tipo[icd_por_tipo['Tipo_elemento_a_buscar'] == tipo]
        ax.plot(data['Porcentaje'], data['ICD'], marker='o', linewidth=2, label=tipo)
    
    ax.set_xlabel('Severidad del daño (%)', fontsize=12)
    ax.set_ylabel('ICD promedio', fontsize=12)
    ax.set_title(f'ICD por Tipo de Elemento - {TIPO_DANO.capitalize()}', fontsize=14, fontweight='bold')
    ax.legend(fontsize=10, title='Tipo de elemento')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(output_dir / f'ICD_por_tipo_elemento_{TIPO_DANO}.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\n✓ Gráfica guardada: ICD_por_tipo_elemento_{TIPO_DANO}.png")
else:
    print("\n⚠️  Columna 'Tipo_elemento_a_buscar' no encontrada. Saltando análisis por tipo.")

## 5. Distribuciones de Componentes del ICD

In [ ]:
# Gráfica de distribuciones
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# DeteccionOK
df_icd['DeteccionOK'].value_counts().sort_index().plot(kind='bar', ax=axes[0,0], color='steelblue')
axes[0,0].set_title('Distribución de DeteccionOK', fontweight='bold')
axes[0,0].set_xlabel('Valor')
axes[0,0].set_ylabel('Frecuencia')
axes[0,0].set_xticklabels(['Fallo (0)', 'Indirecta (0.5)', 'Directa (1)'], rotation=0)

# C_norm (Confianza)
axes[0,1].hist(df_icd['C_norm'], bins=50, color='coral', edgecolor='black', alpha=0.7)
axes[0,1].set_title('Distribución de C_norm (Confianza)', fontweight='bold')
axes[0,1].set_xlabel('C_norm')
axes[0,1].set_ylabel('Frecuencia')

# P_FP (Penalización)
axes[1,0].hist(df_icd['P_FP'], bins=50, color='lightgreen', edgecolor='black', alpha=0.7)
axes[1,0].set_title('Distribución de P_FP (Penalización FP)', fontweight='bold')
axes[1,0].set_xlabel('P_FP')
axes[1,0].set_ylabel('Frecuencia')

# ICD
axes[1,1].hist(df_icd['ICD'], bins=50, color='mediumpurple', edgecolor='black', alpha=0.7)
axes[1,1].set_title('Distribución del ICD', fontweight='bold')
axes[1,1].set_xlabel('ICD')
axes[1,1].set_ylabel('Frecuencia')

plt.tight_layout()
plt.savefig(output_dir / f'distribuciones_componentes_ICD_{TIPO_DANO}.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✓ Gráfica guardada: distribuciones_componentes_ICD_{TIPO_DANO}.png")

## 6. Análisis Binario vs ICD Continuo

In [ ]:
# Crear análisis binario (DeteccionOK > 0 cuenta como éxito)
df_icd['DeteccionOK_binario'] = (df_icd['DeteccionOK'] > 0).astype(int)

# Tasas de detección por severidad
tasa_binaria = df_icd.groupby('Porcentaje')['DeteccionOK_binario'].mean() * 100
icd_normalizado = (df_icd.groupby('Porcentaje')['ICD'].mean() / df_icd['ICD'].max()) * 100

# Comparación
fig, ax = plt.subplots(figsize=(12, 6))

severidades = tasa_binaria.index
ax.plot(severidades, tasa_binaria.values, marker='s', linewidth=2, markersize=8, 
        label='Análisis binario (éxito/fallo)', color='darkblue')
ax.plot(severidades, icd_normalizado.values, marker='o', linewidth=2, markersize=8, 
        label='ICD normalizado (calidad continua)', color='darkred')

ax.set_xlabel('Severidad del daño (%)', fontsize=12)
ax.set_ylabel('Tasa de detección / ICD normalizado (%)', fontsize=12)
ax.set_title(f'Comparación: Binario vs ICD Continuo - {TIPO_DANO.capitalize()}', 
             fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 105])

plt.tight_layout()
plt.savefig(output_dir / f'comparacion_binario_vs_ICD_{TIPO_DANO}.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✓ Gráfica guardada: comparacion_binario_vs_ICD_{TIPO_DANO}.png")

# Tabla comparativa
print(f"\n📊 Comparación numérica:")
comparacion = pd.DataFrame({
    'Severidad (%)': severidades,
    'Tasa Binaria (%)': tasa_binaria.values.round(1),
    'ICD Normalizado (%)': icd_normalizado.values.round(1),
    'Diferencia': (tasa_binaria.values - icd_normalizado.values).round(1)
})
display(comparacion)

## 7. Guardar Dataset con ICD

In [ ]:
# Guardar dataset completo con ICD
output_file = output_dir / f'todos_los_resultados_con_ICD_{TIPO_DANO}.xlsx'
df_icd.to_excel(output_file, index=False, engine='openpyxl')

print(f"\n{'='*70}")
print("RESUMEN FINAL")
print(f"{'='*70}")
print(f"\n✓ Dataset con ICD guardado: {output_file.name}")
print(f"\n📊 Estadísticas generales:")
print(f"  Total de corridas: {len(df_icd)}")
print(f"  ICD promedio: {df_icd['ICD'].mean():.3f} ± {df_icd['ICD'].std():.3f}")
print(f"  ICD mínimo: {df_icd['ICD'].min():.3f}")
print(f"  ICD máximo: {df_icd['ICD'].max():.3f}")
print(f"\n  Detecciones directas: {(df_icd['DeteccionOK'] == 1).sum()} ({100*(df_icd['DeteccionOK'] == 1).sum()/len(df_icd):.1f}%)")
print(f"  Detecciones indirectas: {(df_icd['DeteccionOK'] == 0.5).sum()} ({100*(df_icd['DeteccionOK'] == 0.5).sum()/len(df_icd):.1f}%)")
print(f"  Fallos completos: {(df_icd['DeteccionOK'] == 0).sum()} ({100*(df_icd['DeteccionOK'] == 0).sum()/len(df_icd):.1f}%)")
print(f"\n📁 Archivos generados:")
for file in output_dir.glob('*.png'):
    print(f"  - {file.name}")
print(f"\n{'='*70}")
print("✅ Cálculo de ICD completado exitosamente")
print(f"{'='*70}")

## 📝 Notas

**Interpretación del ICD:**
- **ICD = 0:** Fallo completo (no se detectó el elemento ni nodos cercanos)
- **0 < ICD < 0.5:** Detección de baja calidad (puede ser indirecta o con muchos FPs)
- **0.5 ≤ ICD < 0.8:** Detección de calidad media
- **ICD ≥ 0.8:** Detección de alta calidad (directa, severidad alta, pocos FPs)

**Diferencias clave vs análisis binario:**
- El análisis binario trata todas las detecciones (directas e indirectas) por igual
- El ICD pondera:
  - Detecciones indirectas (0.5) tienen 50% del peso de las directas (1.0)
  - Severidades bajas reducen el ICD
  - Falsos positivos penalizan exponencialmente

**Próximos pasos:**
1. Repetir para corrosión (cambiar `TIPO_DANO = 'corrosion'` en celda 2)
2. Comparar ICD 2026 vs resultados previos
3. Análisis estadístico de vectores alpha (α₁-α₈)
4. Correlación entre ICD y alphas